[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/07_batchnorm_solution.ipynb)

# 🟡 Solution: Implement BatchNorm

*Core Ops & Layers · Medium*

Reference implementation. Try it yourself in `07_batchnorm.ipynb` first.

---
Implement **Batch Normalization** with running statistics, as a pure function.

**Training** — normalise with the current batch's statistics, and move the
running buffers toward them:

$$\mu_{run} \leftarrow (1-m)\,\mu_{run} + m\,\mu_{batch}$$

**Inference** — normalise with the stored buffers, update nothing.

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$$

### Signature
```python
def my_batch_norm(x, gamma, beta, running_mean, running_var,
                  eps=1e-5, momentum=0.1, training=True):
    ...  # -> (out, running_mean, running_var)
```

### Rules
- Do **not** use `nnx.BatchNorm`
- Reduce over the **batch** axis (axis 0), not the feature axis
- Use the **biased** variance (`ddof=0`)
- `momentum=0.1` means the new batch gets weight `0.1`
- Buffers update **only** in training mode

### ⚠️ The one place JAX forces a different signature
PyTorch updates the buffers in place:

```python
running_mean.mul_(1 - momentum).add_(momentum * batch_mean)   # mutates the caller's tensor
```

JAX arrays are **immutable** — there is no `mul_`. So this version **returns**
the new buffers instead:

```python
out, running_mean, running_var = my_batch_norm(x, gamma, beta,
                                               running_mean, running_var)
```

This is not a workaround, it is the JAX model: state is threaded through as
values rather than hidden in objects, which is exactly what makes the function
`jit`-able and free of side effects. (`nnx.BatchStat` exists for when you *do*
want the buffers to live in a module — see how `nnx.BatchNorm` does it.)

### BatchNorm vs LayerNorm
LayerNorm reduces over features, per example — so it is independent of batch
size and works with a batch of 1. BatchNorm reduces over the batch, which
couples examples to each other: predictions change depending on what else was
batched alongside them, and with batch size 1 the variance is 0 and everything
collapses. That coupling is why transformers use LayerNorm.

### A note on the variance convention
This task uses the biased variance for the running buffer, matching
`flax.nnx.BatchNorm`. Real `torch.nn.BatchNorm1d` differs — it updates the
buffer with the *unbiased* variance while normalising with the biased one. The
gap is the Bessel factor $n/(n-1)$, it only shows up at inference, and it is a
classic porting bug. See `jax_pytorch_comparison/crosscheck_vs_torch.py`.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def my_batch_norm(x, gamma, beta, running_mean, running_var,
                  eps=1e-5, momentum=0.1, training=True):
    if training:
        # Reduce over the BATCH axis. Biased variance (ddof=0).
        batch_mean = jnp.mean(x, axis=0)
        batch_var = jnp.var(x, axis=0)

        # PyTorch would do running_mean.mul_(1-m).add_(m*batch_mean) in place.
        # JAX arrays are immutable, so we build new buffers and return them.
        running_mean = (1 - momentum) * running_mean + momentum * batch_mean
        running_var = (1 - momentum) * running_var + momentum * batch_var

        mean, var = batch_mean, batch_var
    else:
        mean, var = running_mean, running_var

    x_norm = (x - mean) / jnp.sqrt(var + eps)
    return gamma * x_norm + beta, running_mean, running_var

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jax.random.normal(jax.random.key(0), (32, 4)) * 3.0 + 5.0
gamma, beta = jnp.ones(4), jnp.zeros(4)
rm, rv = jnp.zeros(4), jnp.ones(4)

out, rm, rv = my_batch_norm(x, gamma, beta, rm, rv, training=True)
print("train-mode column means:", out.mean(0), "(~0)")
print("running_mean after 1 step:", rm, "(moved 10% toward the batch mean)")

eval_out, _, _ = my_batch_norm(x, gamma, beta, rm, rv, training=False)
print("eval-mode column means: ", eval_out.mean(0), "(NOT ~0 — uses the buffers)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("batchnorm")